In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import os
from estnltk import Text
from estnltk.storage.postgres import PostgresStorage, create_schema

from read_config import read_config

In [2]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "v33.db" AS v33')

## koondkorpus

In [3]:
root = os.getcwd()
file_name = "conf_minu.ini"
if os.path.isfile(file_name):
    fname = os.path.basename(file_name).split('/')[-1]
    config = read_config(fname, file_name)
else:
    print("Could not find the specified configuration file.")
    raise SystemExit

In [4]:
source_storage = PostgresStorage(host=config["source_database"]["host"],
                          port=config["source_database"]["port"],
                          dbname=config["source_database"]["database_name"],
                          user=config["source_database"]["username"],
                          password=config["source_database"]["password"],
                          schema=config["source_database"]["work_schema"], 
                          role=config["source_database"]["role"],
                          temporary=False)

INFO:storage.py:57: connecting to host: 'postgres.keeleressursid.ee', port: '5432', dbname: 'estonian-text-corpora', user: 'koljalka'
INFO:storage.py:108: schema: 'estonian_text_corpora', temporary: False, role: 'estonian_text_corpora_read'


In [20]:
source_storage

In [45]:
collection = source_storage['koondkorpus_sentences']

## base tabel kus on verb, kääne, elus_cnt, koht_cnt, distinct root count  

In [4]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1 
"""

source3 = pd.read_sql_query(query, con)
#source3

In [5]:
source3["elus_div"] = source3["elus_cnt"]/source3["root_cnt"]
source3["koht_div"] = source3["koht_cnt"]/source3["root_cnt"]

In [6]:
source3["kaane2"] = ''

source3.loc[source3['kaane'] == 'ill', 'kaane2'] = 'sisse'
source3.loc[source3['kaane'] == 'in', 'kaane2'] = 'sees'
source3.loc[source3['kaane'] == 'el', 'kaane2'] = 'seest'
source3.loc[source3['kaane'] == 'all', 'kaane2'] = 'alale'
source3.loc[source3['kaane'] == 'ad', 'kaane2'] = 'alal'
source3.loc[source3['kaane'] == 'abl', 'kaane2'] = 'alalt'
source3.loc[source3['kaane'] == 'adit', 'kaane2'] = 'sisse'

# võtta konkreetsed näited mõnest kohast

####  näited, mis võiks olla kindlalt koht (rohelised punktid)

In [8]:
source3[(source3["root_cnt"]>=100) & (source3["elus_div"]<=0.05) & (source3["koht_div"]>=0.2)].sort_values(by=["koht_cnt"], ascending=False)

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt,elus_div,koht_div,kaane2
959,kolima,,adit,9,94,427,0.021077,0.220141,sisse
1279,ööbima,,in,8,76,312,0.025641,0.243590,sees
1661,seadma,sisse,in,5,55,237,0.021097,0.232068,sees
1646,askeldama,,in,5,54,239,0.020921,0.225941,sees
1506,lõhkuma,,in,8,53,264,0.030303,0.200758,sees
1596,süütama,,in,11,50,248,0.044355,0.201613,sees
1725,peksma,,in,4,50,227,0.017621,0.220264,sees
2009,tormama,,adit,6,49,190,0.031579,0.257895,sisse
1747,levima,,ill,11,47,225,0.048889,0.208889,sisse
1753,valvama,,in,6,46,225,0.026667,0.204444,sees


### võtame nt kolima + adit

võtta 2-3 näidet lausetest
- kus on koht
- kus on elus
- kus ei ole elus/koht otsust

- vaja saada kätte transaction_head sentence_id

- transaction head join transaction on head_id
- transaction deprel+kääne tingimus

In [51]:
query = """
SELECT 
distinct 
    head.id as head_id,
    head.sentence_id as sentence_id,
    head.verb,
    head.verb_compound as verb_compound,
    --head.loc as verb_loc,
    --tr.loc as root_loc,
    tr.deprel as root_deprel,
    'adit' as kaane,
    tr.lemma as root_lemma,
    tr.koht as koht,
    tr.elus as elus
    
from 
(select *
from transaction_v2
where deprel = 'obl'
and INSTR(',' || feats || ',', ',' || 'adit' || ',') > 0) as tr

join 

(select * from
v33.transaction_head
where 
verb == 'kolima'
and verb_compound == '') as head

on head.id = tr.head_id

"""

r1 = pd.read_sql_query(query, con)
r1

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
0,6815,4016,kolima,,obl,adit,maja,YES,UNK
1,8254,4762,kolima,,obl,adit,riik,YES,UNK
2,11461,6626,kolima,,obl,adit,maja,YES,UNK
3,11764,6782,kolima,,obl,adit,kasiinopealinn,UNK,UNK
4,12443,7230,kolima,,obl,adit,kabinet,YES,UNK
...,...,...,...,...,...,...,...,...,...
1600,29145599,19495889,kolima,,obl,adit,raplane,UNK,UNK
1601,29210285,19633019,kolima,,obl,adit,pealinn,YES,UNK
1602,29426857,20056424,kolima,,obl,adit,kõrvalmaja,YES,UNK
1603,29826341,20918120,kolima,,obl,adit,maja,YES,UNK


In [52]:
r1[r1["elus"]=='YES']

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
43,357977,220129,kolima,,obl,adit,selg,UNK,YES
57,627212,403090,kolima,,obl,adit,lasteaed,YES,YES
72,1011360,637148,kolima,,obl,adit,tiim,UNK,YES
571,9604394,5984655,kolima,,obl,adit,tiim,UNK,YES
629,10668198,6629880,kolima,,obl,adit,käsi,UNK,YES
804,14061211,8760022,kolima,,obl,adit,tall,UNK,YES
824,14404007,8959959,kolima,,obl,adit,tiim,UNK,YES
825,14409203,8962963,kolima,,obl,adit,naiskond,UNK,YES
887,15354835,9565989,kolima,,obl,adit,mets,YES,YES
954,16386388,10211861,kolima,,obl,adit,lasteaed,YES,YES


In [65]:
r1[(r1["elus"]=='UNK') & (r1["koht"]=='UNK')]

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
3,11764,6782,kolima,,obl,adit,kasiinopealinn,UNK,UNK
9,46309,26875,kolima,,obl,adit,Ålborg,UNK,UNK
13,56518,32881,kolima,,obl,adit,teine,UNK,UNK
14,57688,33609,kolima,,obl,adit,küll,UNK,UNK
15,58108,33856,kolima,,obl,adit,Tallinn,UNK,UNK
...,...,...,...,...,...,...,...,...,...
1595,28470447,18852269,kolima,,obl,adit,pool,UNK,UNK
1596,28476561,18856209,kolima,,obl,adit,osariik,UNK,UNK
1598,28586262,18933337,kolima,,obl,adit,küll,UNK,UNK
1599,28715828,19027739,kolima,,obl,adit,Kuressaare,UNK,UNK


# otsida koondkorpusest välja laused kindlalt koht

### näited kohtadest

In [58]:
collection[4016].text #maja

'Suklesed kolisid oma majja 1997. aastal , lõpliku viimistluse sai maja välimus aga alles eelmisel talvel .'

In [59]:
collection[7230].text # kabinet

'Augustis kolib ANNE VEESAAR Vanalinnastuudio direktori kabinetti .'

In [60]:
collection[20918856].text #kant

'kissy: kuhu kanti sa siis kolid'

### näited elusatest

In [61]:
collection[637148].text # tiim

'Vähi , kes eelmised aastad tegeles suuskade libisemisega , kolis šmigunide tiimi ning viis kaasa tarkuse ja info .'

In [62]:
collection[220129].text # selg

'Algselt elutsesid kirbud nahkhiirtel ja pääsukestel , hiljem kolisid samades koobastes elavate inimeste selga .'

In [63]:
collection[14294369].text # rühm

'Oma noored sportlased võib anda ajutiselt kellegi teise hoolde , kuid esiteks on ka ajutisele treenerile vaja lisakoormuse kompenseerimiseks palka ning teiseks kipub nii olema , et edukamad õpilased sellisel ajal lõplikult teise treeneri rühma kolivad .'

### näited mis pole kumbki

In [66]:
collection[18852269].text # pool

'Alumise korruse teise poolde kolis Saare raamatukogu .'

In [67]:
collection[33609].text # küll

'Tema maja Toompuiesteel sai Isamaaliidu kontoriks , tema ise aga kolis perega Kohila kanti , Lohu külla .'

In [68]:
collection[32881].text # teine

'Pätiperiood lõppes , kui Indrek kolis Kuressaare esimesest keskkoolist teise .'

##  näited, mis võiks olla enamuselt koht (kollased punktid)

In [84]:
source3[(source3["root_cnt"]>=100) & (source3["elus_div"]<=0.05) & (source3["koht_div"]>=0.1) & (source3["koht_div"]<0.2)]#.head(10)

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt,elus_div,koht_div,kaane2
218,veetma,,in,22,156,1487,0.014795,0.104909,sees
228,korraldama,,in,25,150,1428,0.017507,0.105042,sees
341,levima,,in,26,110,1077,0.024141,0.102136,sees
374,paiknema,,in,21,125,995,0.021106,0.125628,sees
377,puhkema,,in,21,153,991,0.021191,0.154390,sees
...,...,...,...,...,...,...,...,...,...
3403,võtma,üles,in,1,13,101,0.009901,0.128713,sees
3414,külmuma,,in,3,15,100,0.030000,0.150000,sees
3423,rakenduma,,in,2,12,100,0.020000,0.120000,sees
3424,riivama,,ad,4,15,100,0.040000,0.150000,alal


### võtame nt korraldama + in

võtta 2-3 näidet lausetest
- kus on koht
- kus on elus
- kus ei ole elus/koht otsust

- vaja saada kätte transaction_head sentence_id

- transaction head join transaction on head_id
- transaction deprel+kääne tingimus

In [85]:
query = """
SELECT 
distinct 
    head.id as head_id,
    head.sentence_id as sentence_id,
    head.verb,
    head.verb_compound as verb_compound,
    --head.loc as verb_loc,
    --tr.loc as root_loc,
    tr.deprel as root_deprel,
    'adit' as kaane,
    tr.lemma as root_lemma,
    tr.koht as koht,
    tr.elus as elus
    
from 
(select *
from transaction_v2
where deprel = 'obl'
and INSTR(',' || feats || ',', ',' || 'in' || ',') > 0) as tr

join 

(select * from
v33.transaction_head
where 
verb == 'korraldama'
and verb_compound == '') as head

on head.id = tr.head_id

"""

r1 = pd.read_sql_query(query, con)
r1

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
0,2772,1547,korraldama,,obl,adit,internet,UNK,UNK
1,4788,2865,korraldama,,obl,adit,garaazh,YES,UNK
2,7834,4571,korraldama,,obl,adit,Pärnu,UNK,UNK
3,15801,9212,korraldama,,obl,adit,tele,UNK,UNK
4,27434,15859,korraldama,,obl,adit,Egiptus,UNK,UNK
...,...,...,...,...,...,...,...,...,...
6792,29230774,19675970,korraldama,,obl,adit,priva,UNK,UNK
6793,29318636,19857849,korraldama,,obl,adit,baar,UNK,UNK
6794,29902703,21066763,korraldama,,obl,adit,kodu,YES,UNK
6795,29923267,21100738,korraldama,,obl,adit,noortekas,UNK,UNK


In [91]:
r1[r1["koht"]=='YES'].head(20)

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
1,4788,2865,korraldama,,obl,adit,garaazh,YES,UNK
6,31461,18288,korraldama,,obl,adit,ruum,YES,UNK
17,81020,46989,korraldama,,obl,adit,pealinn,YES,UNK
18,81487,47266,korraldama,,obl,adit,elamu,YES,UNK
23,162476,96136,korraldama,,obl,adit,maa,YES,UNK
28,184091,109397,korraldama,,obl,adit,ruum,YES,UNK
34,216676,129368,korraldama,,obl,adit,kodu,YES,UNK
35,227112,135900,korraldama,,obl,adit,maja,YES,UNK
38,233949,140213,korraldama,,obl,adit,talu,YES,UNK
43,275764,165479,korraldama,,obl,adit,ruum,YES,UNK


In [87]:
r1[r1["elus"]=='YES'].head(20)

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
120,738173,470573,korraldama,,obl,adit,vabariik,UNK,YES
136,824662,523332,korraldama,,obl,adit,vaim,UNK,YES
296,1360790,853741,korraldama,,obl,adit,organism,UNK,YES
304,1392690,874350,korraldama,,obl,adit,kauplus,UNK,YES
534,2105087,1324204,korraldama,,obl,adit,tshehh,UNK,YES
670,2471377,1553135,korraldama,,obl,adit,Vabariik,UNK,YES
811,2875470,1806676,korraldama,,obl,adit,pere,UNK,YES
1013,3569601,2231342,korraldama,,obl,adit,mets,YES,YES
1028,3614108,2258108,korraldama,,obl,adit,mets,YES,YES
1042,3637650,2272271,korraldama,,obl,adit,mets,YES,YES


In [98]:
r1[(r1["elus"]=='UNK') & (r1["koht"]=='UNK')]

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
0,2772,1547,korraldama,,obl,adit,internet,UNK,UNK
2,7834,4571,korraldama,,obl,adit,Pärnu,UNK,UNK
3,15801,9212,korraldama,,obl,adit,tele,UNK,UNK
4,27434,15859,korraldama,,obl,adit,Egiptus,UNK,UNK
5,30447,17704,korraldama,,obl,adit,Edinburgh,UNK,UNK
...,...,...,...,...,...,...,...,...,...
6791,29213349,19639453,korraldama,,obl,adit,msn,UNK,UNK
6792,29230774,19675970,korraldama,,obl,adit,priva,UNK,UNK
6793,29318636,19857849,korraldama,,obl,adit,baar,UNK,UNK
6795,29923267,21100738,korraldama,,obl,adit,noortekas,UNK,UNK


# otsida koondkorpusest välja laused enamuselt koht

### näited kohtadest

In [89]:
collection[2865].text #garaazh

'“ Korraldasime garaazhis väljamüügi , mis on Ameerikas väga levinud võte , ” jutustab Mari-Ann .'

In [92]:
collection[96136].text #maa (mais???????)

'1997.a. mais korraldas linnavalitsus konkursi Narva 1. ja 4. kooli rekonstrueerimiseks .'

In [95]:
collection[331312].text #kool

'1992. a kevadel korraldasid Kirjandusmuuseumi ja Eesti Keele Instituudi folkloristid Eesti koolides tänapäeva õpilasfolkloori kogumise võistluse , mis tõi arhiivi väga suure hulga ( ligi 27 000 lk. ) uut materjali , sh. ligi 3200 “ tavaliste ” mõistatuste teksti ja ligikaudu 25 000 mõistatuste “ perifeeriasse ” kuuluvat teksti .'

### näited elusatest

In [96]:
collection[853741].text # organism

'Põhjamaine toit rahuldab harva organismi kroomivajadust , kroom aga korraldab suhkru ainevahetust organismis ja vähendab magusaisu .'

In [97]:
collection[1806676].text # pere

'Kuid kas ükski keskklassi kuuluv eestlane korraldab avaliku sündmuse rasedusest , sünnitusest ja ristsetest oma peres ?'

In [99]:
collection[3956200].text # süda

'Müncheni südames , Marienplatzil , korraldasid võidurõõmsad sakslased läinudreedese avamängu järel rahvapeo , millele polnuks isegi maailmakuulsal Oktoberfestil midagi vastu panna .'

### näited mis pole kumbki

In [100]:
collection[1547].text # internet

'Aastal 2000 korraldas Valeri Kirss internetis Eesti sajandimissi hääletuse .'

In [101]:
collection[9212].text # tele

'Rahanappuse tõttu kadus ETVst tema menusaade “ Ob-la-di , ob-la-da ” , samal ajal aga korraldab teletäht juba asju Soome teles .'

In [102]:
collection[21100738].text # noortekas

'PETS44: äkki noortekas korraldate oma viktoriine'

# võtta konkreetsed näited mõnest elus verbist

####  näited, mis võiks olla enamus ajast elus (kollased punktid)

In [69]:
source3[(source3["root_cnt"]>=100) & (source3["elus_div"]>=0.15) & (source3["elus_div"]<0.3) & (source3["koht_div"]<=0.05)].sort_values(by=["elus_div"])

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt,elus_div,koht_div,kaane2
2849,siduma,,el,19,4,126,0.150794,0.031746,seest
1703,valima,välja,el,35,8,231,0.151515,0.034632,seest
1614,kuulutama,,el,37,11,244,0.151639,0.045082,seest
1310,järgnema,,el,46,14,303,0.151815,0.046205,seest
2858,jääma,alla,el,19,4,125,0.152000,0.032000,seest
...,...,...,...,...,...,...,...,...,...
498,piisama,,all,232,38,786,0.295165,0.048346,alale
1964,süüdistama,,el,58,6,196,0.295918,0.030612,seest
86,küsima,,abl,795,43,2681,0.296531,0.016039,alalt
1772,pälvima,,abl,66,6,222,0.297297,0.027027,alalt


### võtame nt küsima + abl

võtta 2-3 näidet lausetest
- kus on koht
- kus on elus
- kus ei ole elus/koht otsust

- vaja saada kätte transaction_head sentence_id

- transaction head join transaction on head_id
- transaction deprel+kääne tingimus

In [70]:
query = """
SELECT 
distinct 
    head.id as head_id,
    head.sentence_id as sentence_id,
    head.verb,
    head.verb_compound as verb_compound,
    --head.loc as verb_loc,
    --tr.loc as root_loc,
    tr.deprel as root_deprel,
    'abl' as kaane,
    tr.lemma as root_lemma,
    tr.koht as koht,
    tr.elus as elus
    
from 
(select *
from transaction_v2
where deprel = 'obl'
and INSTR(',' || feats || ',', ',' || 'abl' || ',') > 0) as tr

join 

(select * from
v33.transaction_head
where 
verb == 'küsima'
and verb_compound == '') as head

on head.id = tr.head_id

"""

r1 = pd.read_sql_query(query, con)
r1

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
0,1053,595,küsima,,obl,abl,kaart,UNK,UNK
1,2878,1615,küsima,,obl,abl,mina,UNK,YES
2,3956,2302,küsima,,obl,abl,ise,UNK,UNK
3,4240,2511,küsima,,obl,abl,teineteise,UNK,UNK
4,11575,6671,küsima,,obl,abl,füsioterapeut,UNK,YES
...,...,...,...,...,...,...,...,...,...
11844,30070578,21406748,küsima,,obl,abl,HELEN,UNK,UNK
11845,30074091,21414613,küsima,,obl,abl,teine,UNK,UNK
11846,30074161,21414750,küsima,,obl,abl,mina,UNK,YES
11847,30074163,21414751,küsima,,obl,abl,teine,UNK,UNK


In [77]:
r1[r1["elus"]=='YES']

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
1,2878,1615,küsima,,obl,abl,mina,UNK,YES
4,11575,6671,küsima,,obl,abl,füsioterapeut,UNK,YES
7,17242,10050,küsima,,obl,abl,mina,UNK,YES
9,17906,10478,küsima,,obl,abl,isa,UNK,YES
11,20708,12024,küsima,,obl,abl,mina,UNK,YES
...,...,...,...,...,...,...,...,...,...
11832,30058564,21381349,küsima,,obl,abl,sina,UNK,YES
11835,30060611,21385199,küsima,,obl,abl,arst,UNK,YES
11836,30061974,21388592,küsima,,obl,abl,sina,UNK,YES
11841,30066328,21397893,küsima,,obl,abl,mina,UNK,YES


In [71]:
r1[r1["koht"]=='YES']

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
96,154682,91379,küsima,,obl,abl,keskus,YES,UNK
130,239559,143660,küsima,,obl,abl,riik,YES,UNK
192,416212,263692,küsima,,obl,abl,riik,YES,UNK
253,633451,407091,küsima,,obl,abl,Katus,YES,UNK
343,898087,567638,küsima,,obl,abl,voodi,YES,UNK
...,...,...,...,...,...,...,...,...,...
11163,27841659,18366199,küsima,,obl,abl,riik,YES,UNK
11273,28158904,18610305,küsima,,obl,abl,välismaa,YES,UNK
11359,28494068,18867750,küsima,,obl,abl,riik,YES,UNK
11374,28522445,18886340,küsima,,obl,abl,Kallas,YES,UNK


In [81]:
r1[(r1["elus"]=='UNK') & (r1["koht"]=='UNK')].head(20)

,head_id,sentence_id,verb,verb_compound,root_deprel,kaane,root_lemma,koht,elus
0,1053,595,küsima,,obl,abl,kaart,UNK,UNK
2,3956,2302,küsima,,obl,abl,ise,UNK,UNK
3,4240,2511,küsima,,obl,abl,teineteise,UNK,UNK
5,12634,7360,küsima,,obl,abl,ise,UNK,UNK
6,14389,8377,küsima,,obl,abl,Vahur,UNK,UNK
8,17364,10123,küsima,,obl,abl,Ülo,UNK,UNK
10,19646,11474,küsima,,obl,abl,keegi,UNK,UNK
13,25656,14810,küsima,,obl,abl,poja,UNK,UNK
15,26021,15029,küsima,,obl,abl,ise,UNK,UNK
17,30630,17830,küsima,,obl,abl,tema,UNK,UNK


## otsida koondkorpusest välja laused

### näited kohtadest

In [73]:
collection[567638].text #voodi

'Kuna istusin veel pesus , siis küsisin voodilt : " Kes või mis ma nüüd olen ?'

In [74]:
collection[18610305].text #välismaa

'Minu arvates on normaalne , et me küsime välismaalt nõu ja abi , aga probleem on selles , millisel tasemel ja tasandil nõu ja abi antakse .'

In [75]:
collection[19112689].text # osakond

'Maaelukomisjon küsis arvamust selle eelnõu kohta juriidiliselt osakonnalt , Eesti Maaviljeluse Instituudi taimekaitse sektorilt , Eesti Põllumajandusülikoolilt , Eesti Põllumajandusülikooli Taimekaitse Instituudit , aktsiaseltsilt Kemira , Eesti Konsulentide Ühingult .'

### näited elusatest

In [76]:
collection[6671].text # füsioterapeut

'Jane küsis samas laagris viibinud hispaania füsioterapeudilt , kas too saaks teda aidata ...'

In [78]:
collection[10478].text # isa

"Ta on nimelt veendunud , et tema eelmised abielud Danny Keough' ja Michael Jacksoniga purunesid seetõttu , et ta ei küsinud isalt luba ."

In [79]:
collection[21414750].text # mina

'Rex: miks kõik minult küsivad kus elan ja mu nime MSN-i'

### näited mis pole kumbki

In [80]:
collection[595].text # kaart

"Võimalik , et ta on Tarot' kaartidelt küsinud , millal investeerida ja millal mitte ."

In [82]:
collection[31645].text # vili

'Savisaar räägib , kuidas Vilja ta südame võitis : “ Tema oli selline nooruke , valge kitliga , vist oli valge müts ka ? ” küsib ta Viljalt .'

In [83]:
collection[14810].text # poja

'Küsisin ka pojalt , mida ta arvab , milliseid väärtusi ma olen püüdnud temasse sisendada .'

In [5]:
source_storage.close()

In [6]:
con.close()